# PSU Esports Qwen3.5 Hybrid RAG Test

Notebook นี้ใช้ทดสอบ pipeline ใหม่ในโฟลเดอร์ 19:

- unified corpus
- lexical retrieval
- optional vector retrieval
- RAG + Qwen3.5
- model comparison

## 1. Load Project

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\Chokhun\Downloads\Learn-LLM\19_PSU_Esports_Qwen35_Hybrid_RAG")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.rag.hybrid_engine import HybridRagEngine
from app.rag.ollama_client import ollama_embed

print(PROJECT_ROOT)


C:\Users\Chokhun\Downloads\Learn-LLM\19_PSU_Esports_Qwen35_Hybrid_RAG


## 2. Build Unified Corpus + Lexical Index

In [2]:
import subprocess, sys

subprocess.run([sys.executable, str(PROJECT_ROOT / "tools" / "01_build_unified_corpus.py")], check=True)
subprocess.run([sys.executable, str(PROJECT_ROOT / "tools" / "02_build_lexical_index.py")], check=True)


CompletedProcess(args=['c:\\Users\\Chokhun\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', 'C:\\Users\\Chokhun\\Downloads\\Learn-LLM\\19_PSU_Esports_Qwen35_Hybrid_RAG\\tools\\02_build_lexical_index.py'], returncode=0)

## 3. Ask 1 Question

In [3]:
MODEL = "qwen3:4b"  # เปลี่ยนเป็น "qwen3.5:4b" หลังจาก ollama pull qwen3.5:4b
QUESTION = "สรุปกฎ RoV เรื่อง pause และมาสายให้หน่อย"

engine = HybridRagEngine()
result = engine.answer(
    QUESTION,
    model=MODEL,
    top_k=5,
    use_llm=True,
    timeout_sec=10,
)

print("mode:", result["mode"])
print("model:", result.get("model"))
print("elapsed:", result["elapsed_sec"])
if result.get("error"):
    print("error:", result["error"])
print("-" * 80)
print(result["answer"])
print("-" * 80)
for hit in result["hits"]:
    print(hit["id"], hit["score"], hit["category"], hit["source_kind"])
    print(hit["text_preview"])
    print()


mode: fact_card_synthesis
model: None
elapsed: 0.0752
--------------------------------------------------------------------------------
สรุปจากกติกาที่พบ:
- RoV ถ้าเริ่มแข่งล่าช้าเกิน 15 นาที จะถูกปรับแพ้ในรอบนั้น
- RoV แต่ละทีมสามารถหยุดเกมได้สูงสุด 5 ครั้ง ครั้งละไม่เกิน 1 นาที และเมื่อครบเวลาต้องกลับเข้าแข่งขันต่อ
แหล่งข้อมูล: rov_late_start_forfeit (local://competition_rules/competition_rules_rov_blueket_2025_men); rov_pause_disconnect (local://competition_rules/competition_rules_rov_blueket_2025_men)
--------------------------------------------------------------------------------
rov_late_start_forfeit 4.96129 competition_rules competition_fact_card
RoV ถ้าเริ่มแข่งล่าช้าเกิน 15 นาที จะถูกปรับแพ้ในรอบนั้น หลักฐาน: เอกสารระบุว่าหากเริ่มการแข่งขันล่าช้าเกิน 15 นาที ทีมที่ทำให้เกิดความล่าช้าจะถูกปรับแพ้

rov_pause_disconnect 4.93199 competition_rules competition_fact_card
RoV แต่ละทีมสามารถหยุดเกมได้สูงสุด 5 ครั้ง ครั้งละไม่เกิน 1 นาที และเมื่อครบเวลาต้องกลับเข้าแข่งขันต่อ หลักฐาน: เอ

## 4. Retrieval Only: ดูว่า RAG ดึงอะไรมา

In [4]:
QUESTION = input("พิมพ์คำถามสำหรับเช็ค retrieval: ").strip()

engine = HybridRagEngine()
result = engine.answer(QUESTION, use_llm=False, top_k=8)

print(result["answer"])
print("-" * 80)
for hit in result["hits"]:
    print(hit["id"], "score=", hit["score"], "lexical=", hit["lexical_score"], "vector=", hit["vector_score"])
    print(hit["category"], "/", hit["source_kind"], "/", hit["source_url"])
    print(hit["text_preview"])
    print()


ยังไม่พบข้อมูลที่ยืนยันได้จากฐานข้อมูลที่มีครับ
--------------------------------------------------------------------------------


## 5. Compare Models

In [5]:
QUESTIONS = [
    "CS2 แข่งทีมละกี่คน",
    "สรุปกฎ RoV เรื่อง pause และมาสายให้หน่อย",
    "ต่างมหาลัยเล่น VR 30 นาทีเท่าไหร่",
    "วันไหนหยุดบ้างในเดือนนี้",
    "ทำเมาส์พังต้องเสียค่าปรับไหม",
]

MODELS = ["qwen2.5:3b", "qwen3:4b"]  # เพิ่ม "qwen3.5:4b" หลังโหลดโมเดลแล้ว

engine = HybridRagEngine()
for question in QUESTIONS:
    print("=" * 100)
    print("QUESTION:", question)
    for model in MODELS:
        result = engine.answer(question, model=model, top_k=5, use_llm=True, timeout_sec=10)
        print("-" * 80)
        print("MODEL:", model, "| mode:", result["mode"], "| elapsed:", result["elapsed_sec"])
        if result.get("error"):
            print("error:", result["error"])
        print(result["answer"][:900])


QUESTION: CS2 แข่งทีมละกี่คน
--------------------------------------------------------------------------------
MODEL: qwen2.5:3b | mode: direct_fact | elapsed: 0.0819
CS2 แต่ละทีมประกอบด้วยผู้เล่น 5 คน
แหล่งข้อมูล: local://competition_rules/competition_rules_cs2_psu_phuket_2026
--------------------------------------------------------------------------------
MODEL: qwen3:4b | mode: direct_fact | elapsed: 0.0597
CS2 แต่ละทีมประกอบด้วยผู้เล่น 5 คน
แหล่งข้อมูล: local://competition_rules/competition_rules_cs2_psu_phuket_2026
QUESTION: สรุปกฎ RoV เรื่อง pause และมาสายให้หน่อย
--------------------------------------------------------------------------------
MODEL: qwen2.5:3b | mode: fact_card_synthesis | elapsed: 0.0697
สรุปจากกติกาที่พบ:
- RoV ถ้าเริ่มแข่งล่าช้าเกิน 15 นาที จะถูกปรับแพ้ในรอบนั้น
- RoV แต่ละทีมสามารถหยุดเกมได้สูงสุด 5 ครั้ง ครั้งละไม่เกิน 1 นาที และเมื่อครบเวลาต้องกลับเข้าแข่งขันต่อ
แหล่งข้อมูล: rov_late_start_forfeit (local://competition_rules/competition_rules_rov_blueket_202

## 6. Optional Vector Search

In [7]:
# ต้องรันก่อนใน PowerShell:
# ollama pull qwen3-embedding:0.6b
# py -3 tools\03_build_vector_index_ollama.py --model qwen3-embedding:0.6b

QUESTION = "RoV ถ้ามาสายและหลุดเกมมีกฎยังไง"
EMBEDDING_MODEL = "qwen3-embedding:0.6b"
MODEL = "qwen3:4b"  # เปลี่ยนเป็น qwen3.5:4b ได้

query_embedding = ollama_embed([QUESTION], model=EMBEDDING_MODEL, timeout_sec=30)[0]
engine = HybridRagEngine()
result = engine.answer(
    QUESTION,
    model=MODEL,
    top_k=5,
    query_embedding=query_embedding,
    use_llm=True,
    timeout_sec=10,
)
print(result["answer"])
print("-" * 80)
for hit in result["hits"]:
    print(hit["id"], "score=", hit["score"], "lexical=", hit["lexical_score"], "vector=", hit["vector_score"])


HTTPError: HTTP Error 404: Not Found

## 7. Interactive Ask

In [ ]:
MODEL = "qwen3:4b"  # เปลี่ยนเป็น qwen3.5:4b ได้
engine = HybridRagEngine()

while True:
    q = input("ถามอะไรดี? (exit เพื่อออก): ").strip()
    if q.lower() in {"exit", "quit", "q"}:
        break
    result = engine.answer(q, model=MODEL, top_k=5, use_llm=True, timeout_sec=10)
    print("=" * 80)
    print("mode:", result["mode"], "| elapsed:", result["elapsed_sec"])
    if result.get("error"):
        print("error:", result["error"])
    print(result["answer"])
